# 04 · Anthropic Economic Index panel (Section 3.1.4, Appendix B)

Compiles every AEI release, each with its own schema, into one canonical long-format panel restricted to:
* the `onet_task` facet only (no request, country, SOC-occupation or overall facets),
* global geography only,
* tasks that are in scope per the task master (`subprocess != "OUT_OF_SCOPE"`).

Three steps:
1. **Build** – load each release, map it to the canonical columns, resolve `task_id` by normalised-text match against the task master, apply the scope filter.
2. **Refine** – rename to the final schema (`dimension`, `category`, `task_statement`), shorten the release-specific `variable` prefixes, label plain usage rows `dimension = "usage"`, unpack Release 6 metric ids.
3. **Normalise `pct`** – every `(release, dimension)` group of `pct` rows whose maximum exceeds 1 is on a 0–100 scale and is divided by 100, so `pct` is a 0–1 fraction everywhere.

**Releases.** Raw files go in `inputs/aei/release_1` … `release_6`. Releases 1–5 are present. Release 6 is excluded from the analysis (Appendix B.2) and its raw files are not in the project, so it is skipped; dropping `aei_claude_ai_*.csv` / `aei_1p_api_*.csv` into `inputs/aei/release_6/` adds it automatically. The cost and token indices are **not** re-indexed here; that happens in `final_pipeline` (`r2r_common.build_r2r_indices`).

Run **after 03** (needs `task_master.csv`).

| Output (`outputs/04_aei_panel/`) | Contents |
|---|---|
| `anthropic_panel_onet_global.csv` | step 1: canonical panel (dates, raw entity names, secondary dims) |
| `final_aei_panel.csv` | steps 2–3: analysis panel (Table B2 schema) → **handed off** to `final_pipeline/data/` |
| `build_log.csv` | rows per release at each stage (loaded → task matched → in scope) |
| `pct_rescaled_groups.csv` | which `(release, dimension)` groups were rescaled from 0–100 to 0–1 |
| `panel_coverage.csv` | in-scope tasks with data, by release × platform × dimension × variable (Table B3) |

In [1]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd

import prep_common as pc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## Configuration

In [2]:
RELEASE_DIRS = {r: pc.AEI_DIR / f"release_{r}" for r in range(1, 7)}

# R1/R2 files carry no collection window; R3-R6 dates are read from the files themselves.
# Dates are documentation only: they are dropped from the final panel.
RELEASE_DATES = {
    1: {"date_start": "2024-12-30", "date_end": "2025-01-05"},
    2: {"date_start": "2025-02-25", "date_end": "2025-03-03"},
}

CANONICAL_COLS = ["release", "date_start", "date_end", "platform", "entity_name_raw", "secondary_dim",
                  "secondary_value", "level", "variable", "value", "task_id"]
FINAL_COLS = ["release", "platform", "task_id", "task_statement", "dimension", "category", "level",
              "variable", "value"]

build_log = []


def log_rows(release, stage, df):
    build_log.append({"release": release, "stage": stage, "rows": len(df)})


def find_one(directory, pattern):
    """Exactly one file matching the pattern -- silently picking the wrong file is worse than failing."""
    matches = sorted(glob.glob(str(Path(directory) / pattern)))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one file matching '{pattern}' in {directory}, found {matches}")
    return matches[0]


def warn_if_empty(df, context):
    if len(df) == 0:
        print(f"[WARNING] No rows produced for: {context}. Check the source file's geography/facet values.")

## Step 1 – Build: load each release into the canonical schema

In [3]:
def load_release_1():
    """onet_task_mappings.csv (task_name, pct) -> base task pct. Claude.ai only, no collaboration split."""
    df = pd.read_csv(RELEASE_DIRS[1] / "onet_task_mappings.csv")
    out = pd.DataFrame({"entity_name_raw": df["task_name"], "value": df["pct"]})
    out = out.assign(release=1, platform="claude_ai", secondary_dim=None, secondary_value=None,
                     level=0, variable="pct", task_id=None, **RELEASE_DATES[1])
    return out[CANONICAL_COLS]


def load_release_2():
    """task_pct_v2.csv -> base task pct; automation_vs_augmentation_by_task.csv (wide) -> collaboration."""
    df_pct = pd.read_csv(RELEASE_DIRS[2] / "task_pct_v2.csv")
    base = pd.DataFrame({"entity_name_raw": df_pct["task_name"], "value": df_pct["pct"],
                         "secondary_dim": None, "secondary_value": None, "variable": "pct"})
    df_collab = pd.read_csv(RELEASE_DIRS[2] / "automation_vs_augmentation_by_task.csv")
    melted = (df_collab.melt(id_vars="task_name", value_vars=[c for c in df_collab.columns if c != "task_name"],
                             var_name="secondary_value", value_name="value")
              .rename(columns={"task_name": "entity_name_raw"})
              .assign(secondary_dim="collaboration", variable="pct"))
    out = pd.concat([base, melted[base.columns]], ignore_index=True)
    out = out.assign(release=2, platform="claude_ai", level=0, task_id=None, **RELEASE_DATES[2])
    return out[CANONICAL_COLS]


def _load_long_schema_release(release_num):
    """Releases 3-5: geo_id, geography, date_start, date_end, platform_and_product, facet, level,
    variable, cluster_name, value. facet = 'onet_task' or 'onet_task::<dim>';
    cluster_name = '<task text>' or '<task text>::<category>'."""
    frames = []
    for platform_label, pattern in [("claude_ai", "aei_raw_claude_ai_*.csv"), ("1p_api", "aei_raw_1p_api_*.csv")]:
        path = find_one(RELEASE_DIRS[release_num], pattern)
        df = pd.read_csv(path, low_memory=False)
        df = df[df["geography"] == "global"].copy()
        warn_if_empty(df, f"Release {release_num} / {platform_label}: geography == 'global' ({Path(path).name})")
        if df.empty:
            continue
        df = df[df["facet"].str.startswith("onet_task")].copy()
        df["secondary_dim"] = df["facet"].apply(lambda f: f.split("::")[1] if "::" in f else None)
        split_cols = df["cluster_name"].str.split("::", n=1, expand=True)
        df["entity_name_raw"] = split_cols[0]
        df["secondary_value"] = split_cols[1] if split_cols.shape[1] > 1 else None
        df["platform"] = platform_label
        frames.append(df)
    if not frames:
        return pd.DataFrame(columns=CANONICAL_COLS)
    out = pd.concat(frames, ignore_index=True)
    out["release"] = release_num
    out["task_id"] = None
    return out[CANONICAL_COLS]


# Release 6 schema: date_start, date_end, geo_id, geo_level, category_name, hierarchy_level, metric_id,
# value, node_name, node_external_id. node_external_id is an O*NET element id (a different namespace from
# task_id), so R6 is matched by node_name text like R1-5.
R6_COLLAB_PATTERN_MAP = {
    "collaboration_directive_pct": "directive", "collaboration_feedback_loop_pct": "feedback_loop",
    "collaboration_task_iteration_pct": "task_iteration", "collaboration_validation_pct": "validation",
    "collaboration_learning_pct": "learning", "collaboration_none_pct": "none",
}
R6_COLLAB_BUCKET_MAP = {"collaboration_bucket_automation_pct": "automation",
                        "collaboration_bucket_augmentation_pct": "augmentation"}
R6_USE_CASE_MAP = {"use_case_work_pct": "work", "use_case_personal_pct": "personal",
                   "use_case_coursework_pct": "coursework"}
# multitasking_pct / human_only_ability_pct stay scalar variables: R6 publishes only the "yes" share.


def _r6_parse_metric(metric_id):
    if metric_id in R6_COLLAB_PATTERN_MAP:
        return "pct", "collaboration", R6_COLLAB_PATTERN_MAP[metric_id]
    if metric_id in R6_COLLAB_BUCKET_MAP:
        return "pct", "collaboration_bucket", R6_COLLAB_BUCKET_MAP[metric_id]
    if metric_id in R6_USE_CASE_MAP:
        return "pct", "use_case", R6_USE_CASE_MAP[metric_id]
    return metric_id, None, None


def load_release_6():
    frames = []
    for platform_label, pattern in [("claude_ai", "aei_claude_ai_*.csv"), ("1p_api", "aei_1p_api_*.csv")]:
        for path in sorted(glob.glob(str(RELEASE_DIRS[6] / pattern))):
            df = pd.read_csv(path, low_memory=False)
            df = df[(df["geo_level"] == "global") & (df["geo_id"] == "GLOBAL")]
            df = df[(df["category_name"] == "onet") & (df["hierarchy_level"] == 0)].copy()
            warn_if_empty(df, f"Release 6 / {platform_label} ({Path(path).name})")
            parsed = df["metric_id"].apply(_r6_parse_metric)
            frames.append(pd.DataFrame({
                "release": 6, "date_start": df["date_start"], "date_end": df["date_end"],
                "platform": platform_label, "entity_name_raw": df["node_name"],
                "secondary_dim": parsed.str[1], "secondary_value": parsed.str[2],
                "level": df["hierarchy_level"], "variable": parsed.str[0], "value": df["value"], "task_id": None,
            })[CANONICAL_COLS])
    if not frames:
        print("  Release 6: no raw files in inputs/aei/release_6 -- skipped (excluded from the analysis, Appendix B.2)")
        return pd.DataFrame(columns=CANONICAL_COLS)
    return pd.concat(frames, ignore_index=True)


LOADERS = {1: load_release_1, 2: load_release_2, 3: lambda: _load_long_schema_release(3),
           4: lambda: _load_long_schema_release(4), 5: lambda: _load_long_schema_release(5), 6: load_release_6}
frames = []
for r, loader in LOADERS.items():
    print(f"Loading Release {r} ...")
    df_r = loader()
    print(f"  {len(df_r):,} rows" + (f"  {df_r.groupby('platform').size().to_dict()}" if len(df_r) else ""))
    log_rows(r, "1_loaded (onet_task facet, global)", df_r)
    frames.append(df_r)
panel = pd.concat([f for f in frames if len(f)], ignore_index=True)
print(f"\nCombined (before task-id resolution and scope filter): {len(panel):,} rows")

Loading Release 1 ...
  3,514 rows  {'claude_ai': 3514}
Loading Release 2 ...
  23,549 rows  {'claude_ai': 23549}
Loading Release 3 ...


  46,462 rows  {'1p_api': 26772, 'claude_ai': 19690}
Loading Release 4 ...


  371,063 rows  {'1p_api': 156970, 'claude_ai': 214093}
Loading Release 5 ...


  381,601 rows  {'1p_api': 160582, 'claude_ai': 221019}
Loading Release 6 ...
  Release 6: no raw files in inputs/aei/release_6 -- skipped (excluded from the analysis, Appendix B.2)
  0 rows

Combined (before task-id resolution and scope filter): 826,189 rows


### Resolve `task_id` (normalised-text match) and apply the scope filter

In [4]:
task_master = pc.read_output("task_master", "task_master.csv")

tm_norm = task_master.assign(_norm=task_master["task_statement"].apply(pc.normalize_task_text))
dupes = tm_norm[tm_norm.duplicated("_norm", keep=False)]
if not dupes.empty:
    print(f"[WARNING] {dupes['_norm'].nunique()} normalised task strings map to several task_ids; keeping the first.")
text_to_id = tm_norm.drop_duplicates("_norm", keep="first").set_index("_norm")["task_id"].to_dict()

panel["task_id"] = panel["entity_name_raw"].apply(pc.normalize_task_text).map(text_to_id)
print("Share of rows matched to a register task, by release:")
print(panel.assign(m=panel["task_id"].notna()).groupby("release")["m"].mean().round(4).to_string())
panel = panel[panel["task_id"].notna()].copy()
for r, g in panel.groupby("release"):
    log_rows(r, "2_matched to register task", g)

scope = task_master.set_index("task_id")["subprocess"]
panel["subprocess"] = panel["task_id"].map(scope)
print(f"\n{(panel['subprocess'] == pc.OUT_OF_SCOPE).sum():,} matched rows are OUT_OF_SCOPE and dropped")
panel = panel[panel["subprocess"].notna() & (panel["subprocess"] != pc.OUT_OF_SCOPE)].drop(columns="subprocess")
for r, g in panel.groupby("release"):
    log_rows(r, "3_in scope", g)

panel["task_id"] = panel["task_id"].astype(str)
for c in ["date_start", "date_end"]:
    panel[c] = pd.to_datetime(panel[c], format="%Y-%m-%d", errors="coerce")
panel["release"] = panel["release"].astype(int)
panel["value"] = pd.to_numeric(panel["value"], errors="coerce")
panel = panel.reset_index(drop=True)

print(f"\nCanonical panel: {len(panel):,} rows")
print(panel.groupby(["release", "platform"]).size().rename("rows").to_string())
pc.save_csv(panel, "aei_panel", "anthropic_panel_onet_global.csv")

Share of rows matched to a register task, by release:
release
1    0.0199
2    0.0208
3    0.0215
4    0.0213
5    0.0227

8,588 matched rows are OUT_OF_SCOPE and dropped

Canonical panel: 9,522 rows
release  platform 
1        claude_ai      36
2        claude_ai     273
3        1p_api        284
         claude_ai     208
4        1p_api       1802
         claude_ai    2363
5        1p_api       1952
         claude_ai    2604
[write] outputs/04_aei_panel/anthropic_panel_onet_global.csv  (9,522 rows)


## Step 2 – Refine to the final schema (Table B2)

In [5]:
R6_NUMERIC_DIMS = ["human_only_time", "human_with_ai_time", "ai_autonomy", "human_education_years", "ai_education_years"]
R6_BINARY_SINGLE_DIMS = ["human_only_ability", "multitasking", "task_success"]
# 1p_api dimensions whose category only repeats the variable ("count"/"index") -> category blanked
R1_R5_CATEGORY_REDUNDANT_DIMS = ["completion_tokens", "cost", "prompt_tokens"]


def shorten_r1_r5_variable(variable, original_dim):
    """'onet_task_collaboration_pct' -> 'pct', 'completion_tokens_count' -> 'count', ...
    original_dim is the dimension BEFORE plain usage rows are labelled 'usage'."""
    v = variable
    if v.startswith("onet_task_"):
        v = v[len("onet_task_"):]
    if isinstance(original_dim, str) and original_dim and v.startswith(original_dim + "_"):
        v = v[len(original_dim) + 1:]
    return v


def parse_r6_variable(variable):
    if variable == "pct":
        return "usage", None, "pct"
    if variable.startswith("artifact_") and variable.endswith("_pct"):
        return "artifact", variable[len("artifact_"):-len("_pct")], "pct"
    for dim in R6_NUMERIC_DIMS + R6_BINARY_SINGLE_DIMS:
        if variable.startswith(dim + "_"):
            return dim, None, variable[len(dim) + 1:]
    return None, None, variable


def refine_release_1_2(df):
    df = df.copy()
    df.loc[df["dimension"].isna(), "dimension"] = "usage"
    return df


def refine_release_3_4_5(df):
    df = df.copy()
    original_dim = df["dimension"]
    df["variable"] = [shorten_r1_r5_variable(v, d) for v, d in zip(df["variable"], original_dim)]
    df.loc[df["dimension"].isin(R1_R5_CATEGORY_REDUNDANT_DIMS), "category"] = pd.NA
    df.loc[df["dimension"].isna(), "dimension"] = "usage"
    return df


def refine_release_6(df):
    df = df.copy()
    needs = df["dimension"].isna()
    parsed = df.loc[needs, "variable"].apply(parse_r6_variable)
    df.loc[needs, "dimension"] = [p[0] for p in parsed]
    df.loc[needs, "category"] = [p[1] for p in parsed]
    df.loc[needs, "variable"] = [p[2] for p in parsed]
    unrecognised = df.loc[needs & df["dimension"].isna(), "variable"].value_counts()
    if len(unrecognised):
        print("[WARNING] unrecognised Release 6 variables (left untouched):\n", unrecognised.to_string())
    return df


refined = (panel.drop(columns=["date_start", "date_end"])
           .rename(columns={"secondary_dim": "dimension", "secondary_value": "category",
                            "entity_name_raw": "task_statement"}))
REFINERS = {1: refine_release_1_2, 2: refine_release_1_2, 3: refine_release_3_4_5,
            4: refine_release_3_4_5, 5: refine_release_3_4_5, 6: refine_release_6}
refined = pd.concat([REFINERS[r](g) for r, g in refined.groupby("release", sort=True)], ignore_index=True)
refined = refined[FINAL_COLS]
print(refined.groupby(["release", "dimension"], dropna=False).size().rename("rows").to_string())

release  dimension            
1        usage                     36
2        collaboration            234
         usage                     39
3        collaboration            260
         completion_tokens         44
         cost                      44
         prompt_tokens             44
         usage                    100
4        ai_autonomy              549
         ai_education_years       549
         collaboration            314
         completion_tokens         26
         cost                      26
         human_education_years    549
         human_only_ability       218
         human_only_time          549
         human_with_ai_time       549
         multitasking             214
         prompt_tokens             26
         task_success             216
         usage                    122
         use_case                 258
5        ai_autonomy              594
         ai_education_years       594
         collaboration            352
         completion

## Step 3 – Normalise `pct` to a 0–1 fraction

In [6]:
is_pct = refined["variable"] == "pct"
group_max = refined.loc[is_pct].groupby(["release", "dimension"])["value"].transform("max")
needs_scaling = pd.Series(False, index=refined.index)
needs_scaling.loc[is_pct] = group_max > 1
rescaled = (refined.loc[needs_scaling, ["release", "dimension"]].drop_duplicates()
            .sort_values(["release", "dimension"]).reset_index(drop=True))
refined.loc[needs_scaling, "value"] = refined.loc[needs_scaling, "value"] / 100
print("(release, dimension) groups rescaled from 0-100 to 0-1:")
print(rescaled.to_string(index=False))
pc.save_csv(rescaled, "aei_panel", "pct_rescaled_groups.csv")
pc.save_csv(refined, "aei_panel", "final_aei_panel.csv")

(release, dimension) groups rescaled from 0-100 to 0-1:
 release             dimension
       3         collaboration
       4           ai_autonomy
       4    ai_education_years
       4         collaboration
       4 human_education_years
       4    human_only_ability
       4       human_only_time
       4    human_with_ai_time
       4          multitasking
       4          task_success
       4              use_case
       5           ai_autonomy
       5    ai_education_years
       5         collaboration
       5 human_education_years
       5    human_only_ability
       5       human_only_time
       5    human_with_ai_time
       5          multitasking
       5          task_success
       5              use_case
[write] outputs/04_aei_panel/pct_rescaled_groups.csv  (21 rows)
[write] outputs/04_aei_panel/final_aei_panel.csv  (9,522 rows)


## Build log and coverage (Appendix B.2, Table B3)

In [7]:
log_df = pd.DataFrame(build_log).pivot(index="release", columns="stage", values="rows").fillna(0).astype(int)
pc.save_csv(log_df.reset_index(), "aei_panel", "build_log.csv")
print(log_df.to_string())

coverage = (refined.groupby(["dimension", "variable", "platform", "release"])["task_id"].nunique()
            .unstack("release").fillna(0).astype(int).reset_index())
pc.save_csv(coverage, "aei_panel", "panel_coverage.csv")
coverage

[write] outputs/04_aei_panel/build_log.csv  (6 rows)
stage    1_loaded (onet_task facet, global)  2_matched to register task  3_in scope
release                                                                            
1                                      3514                          70          36
2                                     23549                         490         273
3                                     46462                        1000         492
4                                    371063                        7896        4165
5                                    381601                        8654        4556
6                                         0                           0           0
[write] outputs/04_aei_panel/panel_coverage.csv  (120 rows)


release,dimension,variable,platform,1,2,3,4,5
0,ai_autonomy,count,1p_api,0,0,0,26,28
1,ai_autonomy,count,claude_ai,0,0,0,35,38
2,ai_autonomy,mean,1p_api,0,0,0,26,28
3,ai_autonomy,mean,claude_ai,0,0,0,35,38
4,ai_autonomy,mean_ci_lower,1p_api,0,0,0,26,28
...,...,...,...,...,...,...,...,...
115,usage,pct,claude_ai,36,39,28,35,38
116,use_case,count,1p_api,0,0,0,26,28
117,use_case,count,claude_ai,0,0,0,35,38
118,use_case,pct,1p_api,0,0,0,26,28


## Hand-off to the analysis phase

In [8]:
pc.handoff("aei_panel", "final_aei_panel.csv")

[handoff] outputs/04_aei_panel/final_aei_panel.csv -> ../final_pipeline/data/final_aei_panel.csv
